# MERG — Macro Event Response Gate
## Training Notebook v1.0

Pre-event M1 candlestick anatomy → post-release reaction classifier.

**Architecture:** Two-stage binary classifier (reaction detector → direction classifier).
**Methodology:** Identical to `eurusd_buy_improved.ipynb` — noise-injection voting, purged nested CV, recency-weighted soft-vote ensemble, threshold on validation only, sealed test.

**Dataset:** `ExportedData.csv` — 4,793 rows, 51 columns (45 candlestick features, 6 metadata/target).
**Time range:** 2007-02-13 → 2026-07-31. 183 event types. 0 missing values.

## 0. Configuration

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
from pathlib import Path
from datetime import datetime, timezone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, brier_score_loss
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
# -- Paths --
DATA_CSV = Path(r'C:\Users\david\OneDrive\Documents\fx-prival\ml-signal-service\data\raw\macro\ExportedData.csv')
MODELS_DIR = Path(r'C:\Users\david\OneDrive\Documents\fx-prival\ml-signal-service\models_bin')
STEPS_DIR = Path(r'C:\Users\david\OneDrive\Documents\fx-prival\ml-signal-service\steps\06_merg')
FEATURES_DIR = Path(r'C:\Users\david\OneDrive\Documents\fx-prival\ml-signal-service\data\features')

# -- Split strategy (A = default plan) --
SPLIT_STRATEGY = 'A'

if SPLIT_STRATEGY == 'A':
    TRAIN_END = '2022-12-31'
    VAL_START, VAL_END = '2023-01-01', '2023-12-31'
    TEST_START, TEST_END = '2024-01-01', '2026-07-31'
elif SPLIT_STRATEGY == 'B':
    TRAIN_END = '2019-12-31'
    VAL_START, VAL_END = '2020-01-01', '2022-12-31'
    TEST_START, TEST_END = '2023-01-01', '2026-07-31'
elif SPLIT_STRATEGY == 'C':
    TRAIN_END = '2018-12-31'
    VAL_START, VAL_END = '2019-01-01', '2021-12-31'
    TEST_START, TEST_END = '2022-01-01', '2026-07-31'

# -- Feature selection --
VOTING_PERCENTILE = 40        # percentile above which feature importance counts as a vote
MIN_STRATEGY_SUPPORT = 1      # minimum selection strategies agreeing to keep a feature
N_NOISE_FEATURES = 9          # synthetic noise probes

# -- Model training --
OUTER_FOLDS, INNER_FOLDS = 4, 4
PURGE_DAYS = 30
SEARCH_ITERS = 30
RECENCY_DECAY = 0.15          # exp(-decay * years_ago)
CALIB_CV = 3

# -- Threshold --
MIN_VAL_SIGNALS = 30

print(f'Split strategy: {SPLIT_STRATEGY}')
print(f'Train: pre → {TRAIN_END}')
print(f'Val:   {VAL_START} → {VAL_END}')
print(f'Test:  {TEST_START} → {TEST_END}')

Split strategy: A
Train: pre → 2022-12-31
Val:   2023-01-01 → 2023-12-31
Test:  2024-01-01 → 2026-07-31


## 1. Load & Preprocess

In [3]:
df = pd.read_csv(DATA_CSV, dtype={'event': str, 'time': str})
print(f'Loaded: {df.shape}')
print(f'Missing: {df.isnull().sum().sum()}')
print(f'Events: {df.event.nunique()}')
print(f'Targets: {list(df.columns[-4:])}')

Loaded: (4793, 51)
Missing: 0
Events: 183
Targets: ['target', 'targetSimple', 'target1', 'target2']


In [4]:
# Parse time — assume UTC (confirmed: peak at 15h = 10:30 ET NY window)
df['time_utc'] = pd.to_datetime(df['time'], format='%Y.%m.%d %H:%M', utc=True)
df = df.sort_values('time_utc').reset_index(drop=True)

# Validate time parsing
print(f'Time range: {df.time_utc.min()} → {df.time_utc.max()}')
print(f'Peak hour: {df.time_utc.dt.hour.value_counts().idxmax()} UTC ({df.time_utc.dt.hour.value_counts().max()} events)')
print(f'Chronological: {(df.time_utc.diff().dropna() >= pd.Timedelta(0)).all()}')

Time range: 2007-02-13 13:00:00+00:00 → 2026-07-31 15:30:00+00:00
Peak hour: 15 UTC (1115 events)
Chronological: True


## 2. Feature Engineering

In [5]:
# 45 base candlestick anatomy columns: tWick_i, body_i, bWick_i for i in 15..1
feature_cols = [c for c in df.columns if c.startswith(('tWick', 'body', 'bWick'))]
print(f'Base features: {len(feature_cols)}')

Base features: 45


In [6]:
# Derive microstructure features from the 15-window candlestick anatomy
for i in range(1, 16):
    tw, bd, bw = f'tWick{i}', f'body{i}', f'bWick{i}'

    # Range and components (guarded against zero)
    total_range = df[tw] + df[bd].abs() + df[bw] + 1e-9
    df[f'range_{i}'] = total_range
    df[f'body_ratio_{i}'] = df[bd].abs() / total_range
    df[f'wick_asym_{i}'] = (df[tw] - df[bw]) / total_range
    df[f'body_sign_{i}'] = np.sign(df[bd])  # -1 bearish, 0 doji, +1 bullish

# Cumulative momentum — net drift over the 15-min pre-event window
df['cum_body_1_3'] = df[[f'body{i}' for i in range(1, 4)]].sum(axis=1)
df['cum_body_1_5'] = df[[f'body{i}' for i in range(1, 6)]].sum(axis=1)
df['cum_body_1_15'] = df[[f'body{i}' for i in range(1, 16)]].sum(axis=1)

# Body sign consistency — how many of the last 5 bars share the same sign
df['body_sign_agree_1_5'] = df[[f'body_sign_{i}' for i in range(1, 6)]].sum(axis=1).abs() / 5

# Wick dispersion — max wick asymmetry across recent windows
df['max_wick_asym_1_5'] = df[[f'wick_asym_{i}' for i in range(1, 6)]].abs().max(axis=1)

# Consolidate derived feature list
derived_cols = [c for c in df.columns if c.startswith(('range_', 'body_ratio_', 'wick_asym_',
    'body_sign_', 'cum_body_', 'body_sign_agree_', 'max_wick_asym_'))]
all_feature_cols = feature_cols + derived_cols
print(f'Total features: {len(all_feature_cols)} ({len(feature_cols)} base + {len(derived_cols)} derived)')

Total features: 110 (45 base + 65 derived)


## 3. Label Construction

In [7]:
# Stage 1: binary reaction detector
df['y_reaction'] = (df['targetSimple'] != 'N').astype(int)

# Stage 2: binary direction classifier (only on reaction=1 rows)
df['y_direction'] = np.where(
    df['targetSimple'] == 'U', 1,
    np.where(df['targetSimple'] == 'D', 0, np.nan)
)

print(f'Target distribution:')
print(f'  y_reaction=1 (moved): {df.y_reaction.sum()} ({100*df.y_reaction.mean():.1f}%)')
print(f'  y_reaction=0 (neutral): {(~df.y_reaction.astype(bool)).sum()}')
print(f'  y_direction=1 (UP): {df.y_direction.sum():.0f}')
print(f'  y_direction=0 (DOWN): {(~df.y_direction.astype(bool)).sum():.0f}')

Target distribution:
  y_reaction=1 (moved): 1841 (38.4%)
  y_reaction=0 (neutral): 2952
  y_direction=1 (UP): 943
  y_direction=0 (DOWN): 898


## 4. Chronological Train / Val / Test Split

In [8]:
# Strictly chronological — no shuffling. Future data never leaks into training.
mask_train = df['time_utc'] <= TRAIN_END
mask_val = (df['time_utc'] >= VAL_START) & (df['time_utc'] <= VAL_END)
mask_test = (df['time_utc'] >= TEST_START) & (df['time_utc'] <= TEST_END)

df_train = df[mask_train].copy()
df_val = df[mask_val].copy()
df_test = df[mask_test].copy()

print(f'Train: {len(df_train)} ({df_train.time_utc.min().date()} → {df_train.time_utc.max().date()})')
print(f'Val:   {len(df_val)} ({df_val.time_utc.min().date()} → {df_val.time_utc.max().date()})')
print(f'Test:  {len(df_test)} ({df_test.time_utc.min().date()} → {df_test.time_utc.max().date()})')

# Verify no overlap
assert df_train.index.intersection(df_val.index).empty
assert df_train.index.intersection(df_test.index).empty
assert df_val.index.intersection(df_test.index).empty
print('Split integrity: no overlap ✓')

Train: 3612 (2007-02-13 → 2022-12-23)
Val:   337 (2023-01-03 → 2023-12-22)
Test:  841 (2024-01-03 → 2026-07-30)
Split integrity: no overlap ✓


## 5. Feature Selection — Noise-Injection Voting

Same methodology as `eurusd_buy_improved.ipynb`:
1. Inject synthetic noise probes into train features
2. Train 3 ranking models (RF, LGBM, scaled LogReg) on train only
3. Compute weighted feature importance
4. Vote: keep features above noise percentile with ≥ MIN_STRATEGY_SUPPORT strategy agreement
5. Strip noise probes from final list

In [9]:
X_train_raw = df_train[all_feature_cols].values.astype(np.float32)
y_train = df_train['y_reaction'].values
n_samples, n_features = X_train_raw.shape

In [10]:
# Inject synthetic noise probes (matching notebook method)
rng = np.random.RandomState(42)
noise = np.column_stack([
    rng.normal(0, 1, n_samples),          # Gaussian
    rng.uniform(-1, 1, n_samples),        # Uniform
    rng.poisson(1, n_samples).astype(float),  # Poisson
    np.cumsum(rng.normal(0, 0.1, n_samples)),  # Random walk
    np.sin(np.linspace(0, 10*np.pi, n_samples)),  # Sinusoidal 1
    np.sin(np.linspace(0, 20*np.pi, n_samples)),  # Sinusoidal 2
    rng.exponential(1, n_samples),         # Exponential
    rng.chisquare(3, n_samples).astype(float),  # Chi-squared
    rng.beta(0.5, 0.5, n_samples),         # Beta
])
noise_cols = [f'NOISE_{i}' for i in range(1, N_NOISE_FEATURES + 1)]

X_with_noise = np.column_stack([X_train_raw, noise])
all_feature_names = list(all_feature_cols) + noise_cols
print(f'Features with noise: {X_with_noise.shape[1]} ({len(all_feature_cols)} real + {N_NOISE_FEATURES} noise)')

Features with noise: 119 (110 real + 9 noise)


In [11]:
# Train 3 ranking models on noise-augmented train set (train only — no leakage)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
lgbm = LGBMClassifier(n_estimators=500, random_state=42, verbose=-1)
lr = LogisticRegression(max_iter=5000, random_state=42)

rf.fit(X_with_noise, y_train)
lgbm.fit(X_with_noise, y_train)
lr.fit(X_with_noise, y_train)

# Weighted importance: 0.3·RF + 0.6·LGBM + 0.1·LogReg (abs coef, scaled)
rf_imp = rf.feature_importances_
lgbm_imp = lgbm.feature_importances_
lr_coef = np.abs(lr.coef_[0])
lr_imp = lr_coef / (lr_coef.sum() + 1e-9)

avg_imp = 0.3 * rf_imp + 0.6 * lgbm_imp + 0.1 * lr_imp
noise_mask = np.array([n.startswith('NOISE_') for n in all_feature_names])
noise_imp = avg_imp[noise_mask]
real_imp = avg_imp[~noise_mask]

print(f'Noise importance range: {noise_imp.min():.6f} — {noise_imp.max():.6f}')
print(f'Real feature importance range: {real_imp.min():.6f} — {real_imp.max():.6f}')
print(f'Features beating best noise: {(real_imp > noise_imp.max()).sum()}')

Noise importance range: 9.000411 — 100.204588
Real feature importance range: 0.000410 — 450.610890
Features beating best noise: 25


In [12]:
# Voting: each model votes if importance > its own VOTING_PERCENTILE
percentile_thresh = np.percentile(avg_imp, VOTING_PERCENTILE)
rf_votes = rf_imp > np.percentile(rf_imp, VOTING_PERCENTILE)
lgbm_votes = lgbm_imp > np.percentile(lgbm_imp, VOTING_PERCENTILE)
lr_votes = lr_imp > np.percentile(lr_imp, VOTING_PERCENTILE)
total_votes = rf_votes.astype(int) + lgbm_votes.astype(int) + lr_votes.astype(int)

# Selection strategies (matching notebook)
noise_quantile_70 = np.percentile(avg_imp[noise_mask], 70)
s1 = real_imp > noise_imp.max()                    # beat best noise
s2 = real_imp > noise_quantile_70                  # beat noise p70
s3 = total_votes[~noise_mask] > total_votes[noise_mask].max()  # more votes than best noise
s4 = real_imp > avg_imp[noise_mask].mean()         # beat noise mean
s5 = (total_votes[~noise_mask] >= 1) & (real_imp > avg_imp[noise_mask].mean())

strategy_support = s1.astype(int) + s2.astype(int) + s3.astype(int) + s4.astype(int) + s5.astype(int)
selected_mask = strategy_support >= MIN_STRATEGY_SUPPORT

selected_features = [all_feature_cols[i] for i in range(len(all_feature_cols)) if selected_mask[i]]
print(f'Selected: {len(selected_features)} / {len(all_feature_cols)} features')
print(f'Dropped: {len(all_feature_cols) - len(selected_features)} (below noise threshold)')
print(f'\nTop 10 by importance:')
for rank, (feat, imp) in enumerate(sorted(zip(selected_features, real_imp[selected_mask]),
    key=lambda x: -x[1])[:10], 1):
    print(f'  {rank:2d}. {feat:<25s} imp={imp:.4f}  votes={total_votes[~noise_mask][selected_mask][rank-1]}')

NameError: name 'noise_quantile_70' is not defined

## 6. Model Training — Two-Stage Classifier

### Stage 1 — Reaction Detector (binary: will it move?)

In [ ]:
def make_recency_weights(dates, decay=RECENCY_DECAY):
    """Exponential recency weighting — recent samples get higher weight."""
    years_ago = (dates.max() - dates).dt.days / 365.25
    w = np.exp(-decay * years_ago)
    return w / w.sum() * len(w)

def fit_stage(X, y, dates, stage_name='Stage1'):
    """
    Fit two-stage model with nested purged CV, recency weighting,
    and soft-vote ensemble of 4 calibrated classifiers.
    Returns: dict with {'ensemble': VotingClassifier, 'features': list, 'threshold': float}
    """
    sample_weight = make_recency_weights(dates)
    
    outer_cv = TimeSeriesSplit(n_splits=OUTER_FOLDS)
    inner_cv = TimeSeriesSplit(n_splits=INNER_FOLDS)
    
    # Candidates
    models = {
        'LogReg': LogisticRegression(max_iter=5000, random_state=42),
        'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        'XGBoost': XGBClassifier(scale_pos_weight=len(y)/y.sum(), random_state=42, verbosity=0),
        'LightGBM': LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1),
    }
    
    pr_scores = {}
    for name, model in models.items():
        fold_scores = []
        for train_idx, test_idx in outer_cv.split(X):
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]
            w_tr = sample_weight[train_idx]
            model.fit(X_tr, y_tr, sample_weight=w_tr)
            proba = model.predict_proba(X_te)[:, 1]
            precision, recall, _ = precision_recall_curve(y_te, proba)
            fold_scores.append(auc(recall, precision))
        pr_scores[name] = np.mean(fold_scores)
        print(f'  {name:<15s} PR-AUC: {pr_scores[name]:.4f}')
    
    best_name = max(pr_scores, key=pr_scores.get)
    print(f'  Best: {best_name} (PR-AUC={pr_scores[best_name]:.4f})')
    
    # Final ensemble — soft-vote of all 4 calibrated models
    from sklearn.ensemble import VotingClassifier
    estimators = []
    for name, model in models.items():
        cal = CalibratedClassifierCV(model, method='isotonic', cv=CALIB_CV)
        estimators.append((name, cal))
    
    ensemble = VotingClassifier(estimators=estimators, voting='soft')
    ensemble.fit(X, y, sample_weight=sample_weight)
    
    return ensemble, pr_scores

In [ ]:
# Stage 1 — reaction detector
X_train = df_train[selected_features].values.astype(np.float32)
y_train_s1 = df_train['y_reaction'].values
dates_train = df_train['time_utc']

X_val = df_val[selected_features].values.astype(np.float32)
y_val_s1 = df_val['y_reaction'].values

print('=== Stage 1 — Reaction Detector ===')
ensemble_s1, pr_s1 = fit_stage(X_train, y_train_s1, dates_train, 'Stage1')
print()

In [ ]:
# Threshold calibration on validation only
proba_val_s1 = ensemble_s1.predict_proba(X_val)[:, 1]
precision_vals, recall_vals, thresholds = precision_recall_curve(y_val_s1, proba_val_s1)

# Scan thresholds with minimum-signal floor
best_thresh_s1, best_score_s1 = 0.5, -np.inf
for i, t in enumerate(thresholds):
    n_signals = (proba_val_s1 >= t).sum()
    if n_signals < MIN_VAL_SIGNALS:
        continue
    if i >= len(precision_vals):
        continue
    score = precision_vals[i] * np.log(n_signals + 1)
    if score > best_score_s1:
        best_score_s1 = score
        best_thresh_s1 = t

print(f'Stage 1 optimal threshold: {best_thresh_s1:.4f} (score={best_score_s1:.4f})')
print(f'Signals at threshold: {(proba_val_s1 >= best_thresh_s1).sum()} / {len(proba_val_s1)}')

### Stage 2 — Direction Classifier (conditional on reaction)

In [ ]:
# Stage 2 trains only on rows where a reaction occurred
mask_react_train = df_train['y_reaction'] == 1
mask_react_val = df_val['y_reaction'] == 1

X_train_s2 = df_train.loc[mask_react_train, selected_features].values.astype(np.float32)
y_train_s2 = df_train.loc[mask_react_train, 'y_direction'].values
dates_train_s2 = df_train.loc[mask_react_train, 'time_utc']

X_val_s2 = df_val.loc[mask_react_val, selected_features].values.astype(np.float32)
y_val_s2 = df_val.loc[mask_react_val, 'y_direction'].values

print(f'Stage 2 train: {len(y_train_s2)} samples (UP={y_train_s2.sum():.0f}, DOWN={(~y_train_s2.astype(bool)).sum():.0f})')
print(f'Stage 2 val:   {len(y_val_s2)} samples')
print()
print('=== Stage 2 — Direction Classifier ===')
ensemble_s2, pr_s2 = fit_stage(X_train_s2, y_train_s2, dates_train_s2, 'Stage2')
print()

In [ ]:
# Threshold calibration on validation only
proba_val_s2 = ensemble_s2.predict_proba(X_val_s2)[:, 1]
precision_vals2, recall_vals2, thresholds2 = precision_recall_curve(y_val_s2, proba_val_s2)

best_thresh_s2, best_score_s2 = 0.5, -np.inf
for i, t in enumerate(thresholds2):
    n_signals = (proba_val_s2 >= t).sum()
    if n_signals < MIN_VAL_SIGNALS:
        continue
    if i >= len(precision_vals2):
        continue
    score = precision_vals2[i] * np.log(n_signals + 1)
    if score > best_score_s2:
        best_score_s2 = score
        best_thresh_s2 = t

print(f'Stage 2 optimal threshold: {best_thresh_s2:.4f} (score={best_score_s2:.4f})')
print(f'Signals at threshold: {(proba_val_s2 >= best_thresh_s2).sum()} / {len(proba_val_s2)}')

## 7. Evaluation — Sealed Test Set

In [ ]:
# Test set — touched exactly once, at the very end
X_test = df_test[selected_features].values.astype(np.float32)
y_test_s1 = df_test['y_reaction'].values
mask_react_test = df_test['y_reaction'] == 1
X_test_s2 = df_test.loc[mask_react_test, selected_features].values.astype(np.float32)
y_test_s2 = df_test.loc[mask_react_test, 'y_direction'].values

# Stage 1 evaluation
proba_test_s1 = ensemble_s1.predict_proba(X_test)[:, 1]
pred_test_s1 = (proba_test_s1 >= best_thresh_s1).astype(int)

roc_s1 = roc_auc_score(y_test_s1, proba_test_s1)
prec_s1, rec_s1, _ = precision_recall_curve(y_test_s1, proba_test_s1)
pr_auc_s1 = auc(rec_s1, prec_s1)
brier_s1 = brier_score_loss(y_test_s1, proba_test_s1)

print('=== Stage 1 — Reaction Detector (Test) ===')
print(f'  ROC-AUC:  {roc_s1:.4f}')
print(f'  PR-AUC:   {pr_auc_s1:.4f}')
print(f'  Brier:    {brier_s1:.4f}')
print(f'  Signals:  {pred_test_s1.sum()} / {len(pred_test_s1)} ({100*pred_test_s1.mean():.1f}%)')

In [ ]:
# Stage 2 evaluation (on reaction subset only)
if len(X_test_s2) > 0:
    proba_test_s2 = ensemble_s2.predict_proba(X_test_s2)[:, 1]
    pred_test_s2 = (proba_test_s2 >= best_thresh_s2).astype(int)

    roc_s2 = roc_auc_score(y_test_s2, proba_test_s2)
    prec_s2, rec_s2, _ = precision_recall_curve(y_test_s2, proba_test_s2)
    pr_auc_s2 = auc(rec_s2, prec_s2)
    brier_s2 = brier_score_loss(y_test_s2, proba_test_s2)

    print('=== Stage 2 — Direction Classifier (Test) ===')
    print(f'  ROC-AUC:  {roc_s2:.4f}')
    print(f'  PR-AUC:   {pr_auc_s2:.4f}')
    print(f'  Brier:    {brier_s2:.4f}')
    print(f'  Samples:  {len(y_test_s2)}')
else:
    roc_s2, pr_auc_s2, brier_s2 = np.nan, np.nan, np.nan
    print('Stage 2: no reaction samples in test set')

In [ ]:
# Combined runtime simulation on test set
proba_reaction = proba_test_s1
proba_up = np.zeros(len(proba_reaction))
if len(X_test_s2) > 0:
    proba_up[mask_react_test.values] = ensemble_s2.predict_proba(X_test_s2)[:, 1]

p_U = proba_reaction * proba_up
p_D = proba_reaction * (1 - proba_up)
p_N = 1 - proba_reaction

# Predicted class = argmax
probs = np.column_stack([p_U, p_D, p_N])
pred_class_idx = np.argmax(probs, axis=1)
pred_conf = np.max(probs, axis=1)
class_map = {0: 'U', 1: 'D', 2: 'N'}
pred_class = [class_map[i] for i in pred_class_idx]

actual = df_test['targetSimple'].values
correct = np.array(pred_class) == actual
print(f'Combined accuracy: {correct.mean():.4f}')
print(f'Majority baseline (always N): {100*(actual=="N").mean():.1f}%')
print(f'\nConfusion matrix (predicted → actual):')
for p in ['N', 'U', 'D']:
    row = []
    for a in ['N', 'U', 'D']:
        cnt = ((np.array(pred_class) == p) & (actual == a)).sum()
        row.append(f'{cnt:4d}')
    print(f'  pred_{p}: ' + '  '.join(row))

## 8. Serialize Model Bundles

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Stage 1 bundle
bundle_s1 = {
    'model': ensemble_s1,
    'features': selected_features,
    'threshold': best_thresh_s1,
    'roc_auc': float(roc_s1),
    'pr_auc': float(pr_auc_s1),
    'brier': float(brier_s1),
    'split_strategy': SPLIT_STRATEGY,
    'train_end': TRAIN_END,
    'trained_at': datetime.now(timezone.utc).isoformat(),
}
joblib.dump(bundle_s1, MODELS_DIR / 'MERG_v1_reaction.joblib')
print(f'Saved MERG_v1_reaction.joblib ({len(selected_features)} features)')

# Stage 2 bundle
bundle_s2 = {
    'model': ensemble_s2,
    'features': selected_features,
    'threshold': best_thresh_s2,
    'roc_auc': float(roc_s2) if not np.isnan(roc_s2) else None,
    'pr_auc': float(pr_auc_s2) if not np.isnan(pr_auc_s2) else None,
    'brier': float(brier_s2) if not np.isnan(brier_s2) else None,
    'split_strategy': SPLIT_STRATEGY,
    'train_end': TRAIN_END,
    'trained_at': datetime.now(timezone.utc).isoformat(),
}
joblib.dump(bundle_s2, MODELS_DIR / 'MERG_v1_direction.joblib')
print(f'Saved MERG_v1_direction.joblib')

# Feature list artifact
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
with open(FEATURES_DIR / 'merg_v1_features.txt', 'w') as f:
    for feat in selected_features:
        f.write(feat + '\n')
print(f'Saved merg_v1_features.txt')

print('\n=== Training complete ===')
print(f'Stage 1 ROC-AUC: {roc_s1:.4f} (target: >= 0.58)')
print(f'Stage 2 ROC-AUC: {roc_s2:.4f} (target: >= 0.55)' if not np.isnan(roc_s2) else 'Stage 2: insufficient test samples')
print(f'Features: {len(selected_features)} selected from {len(all_feature_cols)} candidates')

---
*Notebook complete. Deploy MERG_v1_reaction.joblib and MERG_v1_direction.joblib to models_bin/.*